In [3]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings

warnings.filterwarnings("ignore")

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [4]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[**********            21%                       ]  105 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[***************       32%                       ]  158 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************56%**                     ]  278 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['VYLR', 'HONA', 'FDXF']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### Volume crossover strategy using VWAP (Volume Weighted Average Price)

In [5]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
vwap_dict = {}

for ticker in close_df.columns:

    prices = close_df[ticker].to_numpy(dtype=np.float64)
    low = low_df[ticker].to_numpy(dtype=np.float64)
    high = high_df[ticker].to_numpy(dtype=np.float64)
    volume = volume_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    vwap = talib.VWAP(high=high, low=low, close=prices, volume=volume)

    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    vwap_dict[ticker] = vwap

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
vwap_df = pd.DataFrame(vwap_dict, index = close_df.index)

entries = ((close_df > vwap_df) &
    (close_df.shift(1) <= vwap_df.shift(1))).shift(1).fillna(False).astype(bool)

exits = ((close_df < vwap_df) &
    (close_df.shift(1) >= vwap_df.shift(1))).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               208842.452764
Total Return [%]                           108.842453
Benchmark Return [%]                       524.559314
Max Gross Exposure [%]                      99.998931
Total Fees Paid                           54229.13715
Max Drawdown [%]                            22.549755
Max Drawdown Duration               671 days 00:00:00
Total Trades                                    10855
Total Closed Trades                             10838
Total Open Trades                                  17
Open Trade PnL                             961.597598
Win Rate [%]                                36.713416
Best Trade [%]                              50.155578
Worst Trade [%]                            -30.036115
Avg Winning Trade [%]       

In [6]:
entries = ((close_df > vwap_df) &
    (close_df.shift(1) <= vwap_df.shift(1)) & (rsi_df < 30)).shift(1).fillna(False).astype(bool)

exits = ((close_df < vwap_df) &
    (close_df.shift(1) >= vwap_df.shift(1)) & (rsi_df > 70)).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
    tp_stop = 0.10
       
)


print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               103135.798723
Total Return [%]                             3.135799
Benchmark Return [%]                       524.559314
Max Gross Exposure [%]                      56.349292
Total Fees Paid                           1796.671393
Max Drawdown [%]                            12.669582
Max Drawdown Duration              1126 days 00:00:00
Total Trades                                      104
Total Closed Trades                               104
Total Open Trades                                   0
Open Trade PnL                                    0.0
Win Rate [%]                                54.807692
Best Trade [%]                              17.776136
Worst Trade [%]                            -21.687321
Avg Winning Trade [%]       